In [1]:
import os, glob, unicodedata
import pandas as pd
import numpy as np

d = '../datos/raw/'

def limpiar_nombre(nombre):
    if pd.isna(nombre):
        return ""
    s = ''.join(c for c in unicodedata.normalize('NFD', str(nombre)) if unicodedata.category(c) != 'Mn')
    return s.strip().lower()

def cargar_fbref(tipo):
    lista = []
    for f in sorted(glob.glob(f'{d}fbref_{tipo}_*.csv')):
        s = os.path.basename(f).replace(f'fbref_{tipo}_', '').replace('.csv', '')
        x = pd.read_csv(f, header=1, on_bad_lines='skip')
        x = x[x['Player'].notna() & (x['Player'] != 'Player')].copy()
        x['Season'] = s
        lista.append(x)
    return pd.concat(lista, ignore_index=True)

In [2]:
fb = cargar_fbref('std')
for c in ['Min', 'Gls', 'Ast']:
    fb[c] = pd.to_numeric(fb[c].astype(str).str.replace(',', ''), errors='coerce')
fb = fb.dropna(subset=['Min'])
fb['Player_Clean'] = fb['Player'].apply(limpiar_nombre)

# En tus archivos Gls y Ast vienen por 90: reconstruimos los totales
fb['G_tot'] = (fb['Gls'] * fb['Min'] / 90).round()
fb['A_tot'] = (fb['Ast'] * fb['Min'] / 90).round()
print(len(fb), 'filas FBref;', fb['Season'].nunique(), 'temporadas')

17178 filas FBref; 6 temporadas


In [3]:
lista = []
# Buscará cualquier archivo que empiece con "under_"
for f in sorted(glob.glob(d + 'under_*.csv')):
    # AQUÍ ESTÁ LA CORRECCIÓN: Extrae la temporada correctamente siempre
    s = os.path.basename(f).split('_')[-1].replace('.csv', '')
    x = pd.read_csv(f, sep=None, engine='python')
    x['Season'] = s
    lista.append(x)
    
us = pd.concat(lista, ignore_index=True)
us['Player_Clean'] = us['player'].apply(limpiar_nombre)

print(us.columns.tolist())
print('Duplicados jugador-temporada en Understat:', us.duplicated(['Player_Clean', 'Season']).sum())

['\ufeff"number"', 'player', 'team', 'apps', 'min', 'goals', 'NPG', 'a', 'xG', 'NPxG', 'xA', 'xGChain', 'xGBuildup', 'xG90', 'NPxG90', 'xA90', 'xG90xA90', 'NPxG90xA90', 'xGChain90', 'xGBuildup90', 'Season', 'Player_Clean']
Duplicados jugador-temporada en Understat: 462


In [4]:
es_fw = fb['Pos'].str.startswith('FW', na=False)
fw_keys = set(zip(fb.loc[es_fw, 'Player_Clean'], fb.loc[es_fw, 'Born']))

agg = (fb.groupby(['Player_Clean', 'Born', 'Season'], as_index=False, dropna=False)
         .agg(Player=('Player', 'first'),
              Squad=('Squad', lambda x: '/'.join(x)),
              Min=('Min', 'sum'),
              G_tot=('G_tot', 'sum'),
              A_tot=('A_tot', 'sum')))

agg['es_fw'] = [(p, b) in fw_keys for p, b in zip(agg['Player_Clean'], agg['Born'])]
agg = agg[agg['es_fw'] & (agg['Min'] >= 900)].copy()
print('Registros jugador-temporada (delanteros, >=900 min):', len(agg))

Registros jugador-temporada (delanteros, >=900 min): 2191


In [5]:
equivalencias = {
    'emi buendia': 'emiliano buendia',
    'francisco trincao': 'trincao',
    'hwang hee-chan': 'hee-chan hwang',
    'igor thiago': 'thiago',
    'josh sargent': 'joshua sargent',
}

agg['Player_Clean_fbref'] = agg['Player_Clean']
agg['Player_Clean'] = agg['Player_Clean'].replace(equivalencias)

cruce = agg.merge(us, on=['Player_Clean', 'Season'], how='left', indicator=True)
print(cruce['_merge'].value_counts())
print('Filas en agg:', len(agg), '| filas tras el cruce:', len(cruce))

sin_cruce = cruce[cruce['_merge'] == 'left_only']
print(sin_cruce[['Player', 'Squad', 'Season', 'Min']].to_string())

_merge
both          2099
left_only      202
right_only       0
Name: count, dtype: int64
Filas en agg: 2191 | filas tras el cruce: 2301
                      Player                      Squad Season   Min
3            Abde Ezzalzouli                    Osasuna  22-23  1504
4            Abde Ezzalzouli       Barcelona/Real Betis  23-24   933
5            Abde Ezzalzouli                 Real Betis  24-25  1984
6            Abde Ezzalzouli                 Real Betis  25-26  2247
35              Adri Embarba                   Espanyol  21-22  1986
36              Adri Embarba           Almería/Espanyol  22-23  1507
37              Adri Embarba                    Almería  23-24  2476
38              Adri Embarba             Rayo Vallecano  24-25  1607
52        Albert Guðmundsson                      Genoa  23-24  3016
53        Albert Guðmundsson                 Fiorentina  24-25  1273
54        Albert Guðmundsson                 Fiorentina  25-26  2250
76          Alexandre Alemão       

In [6]:
ok = cruce.copy()
ok['ratio_min'] = ok['min'] / ok['Min']
ok['dif_goles'] = (ok['goals'] - ok['G_tot']).abs()

sospechosos = ok[(ok['ratio_min'] < 0.85) | (ok['ratio_min'] > 1.15) | (ok['dif_goles'] > 3)]
print(len(sospechosos), 'registros a revisar de', len(ok))
print(sospechosos[['Player', 'Squad', 'team', 'Season', 'Min', 'min', 'G_tot', 'goals']].to_string())

168 registros a revisar de 2301
                         Player                         Squad                     team Season   Min     min  G_tot  goals
18                   Adam Ounas                  Lille/Napoli                   Napoli  22-23  1079     7.0    1.0    0.0
20                 Adama Traoré              Barcelona/Wolves  Wolverhampton Wanderers  21-22  1449  1047.0    1.0    1.0
21                 Adama Traoré              Barcelona/Wolves                Barcelona  21-22  1449   375.0    1.0    0.0
29              Ademola Lookman      Atalanta/Atlético Madrid          Atletico Madrid  25-26  1412   638.0    6.0    4.0
30              Ademola Lookman      Atalanta/Atlético Madrid                 Atalanta  25-26  1412   803.0    6.0    2.0
43                   Akor Adams           Montpellier/Sevilla                  Sevilla  24-25  1209   122.0    3.0    0.0
66               Alexander Isak       Newcastle/Real Sociedad            Real Sociedad  22-23  1674   157.0   11.0

In [7]:
cols = ['Player', 'Player_Clean', 'Born', 'Season', 'Squad', 'Min', 'G_tot', 'A_tot',
        'team', 'min', 'goals', 'NPG', 'a', 'xG', 'NPxG', 'xA', 'xG90', 'NPxG90', 'xA90']
cols = [c for c in cols if c in cruce.columns]

base = cruce[cols].copy()
os.makedirs('../datos/procesado', exist_ok=True)
base.to_csv('../datos/procesado/fw_base.csv', index=False)
print(base.shape)
display(base.head())
sht = cargar_fbref('sht')
print(sht.columns.tolist())
print(sht[sht['Player'] == 'Mohamed Salah'].head(3).to_string())

(2301, 19)


,Player,Player_Clean,Born,Season,Squad,Min,G_tot,A_tot,team,min,goals,NPG,a,xG,NPxG,xA,xG90,NPxG90,xA90
0,Aaron Leya Iseka,aaron leya iseka,1997.0,20-21,Metz,1008,4.0,0.0,Metz,1015.0,4.0,2.0,0.0,4.40,2.12,0.61,0.39,0.19,0.05
1,Abdallah Sima,abdallah sima,2001.0,22-23,Angers,1836,5.0,1.0,Angers,1824.0,5.0,5.0,1.0,4.46,4.46,1.81,0.22,0.22,0.09
2,Abdallah Sima,abdallah sima,2001.0,24-25,Brest,1238,7.0,2.0,Brest,1229.0,6.0,6.0,2.0,7.70,7.70,1.70,0.56,0.56,0.12
3,Abde Ezzalzouli,abde ezzalzouli,2001.0,22-23,Osasuna,1504,4.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Abde Ezzalzouli,abde ezzalzouli,2001.0,23-24,Barcelona/Real Betis,933,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', '90s', 'Gls', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'PK', 'PKatt', 'Matches', '-9999', 'Season']
        Rk         Player  Nation Pos      Squad                Comp   Age    Born   90s   Gls    Sh   SoT  SoT%  Sh/90  SoT/90  G/Sh  G/SoT    PK  PKatt  Matches     -9999 Season
2306  2307  Mohamed Salah  eg EGY  FW  Liverpool  eng Premier League  28.0  1992.0  34.2  0.64  3.68  1.58  42.9   3.68    1.58  0.17   0.41  0.18   0.18  Matches  e342ad68  20-21
5178  2357  Mohamed Salah  eg EGY  FW  Liverpool  eng Premier League  29.0  1992.0  30.7  0.75  4.53  1.95  43.2   4.53    1.96  0.17   0.38  0.16   0.20  Matches  e342ad68  21-22
8077  2334  Mohamed Salah  eg EGY  FW  Liverpool  eng Premier League  30.0  1992.0  36.6  0.52  3.42  1.28  37.6   3.42    1.29  0.15   0.40  0.05   0.11  Matches  e342ad68  22-23


In [8]:
sht = cargar_fbref('sht')
for c in ['90s', 'Sh', 'SoT']:
    sht[c] = pd.to_numeric(sht[c], errors='coerce')
sht['Player_Clean'] = sht['Player'].apply(limpiar_nombre)

# Totales reconstruidos desde las tasas por 90
sht['Sh_tot'] = (sht['Sh'] * sht['90s']).round()
sht['SoT_tot'] = (sht['SoT'] * sht['90s']).round()

sht['Player_Clean'] = sht['Player_Clean'].replace(equivalencias)

sht_agg = (sht.groupby(['Player_Clean', 'Born', 'Season'], as_index=False, dropna=False)
              .agg(Sh_tot=('Sh_tot', 'sum'), SoT_tot=('SoT_tot', 'sum')))

In [9]:
base = cruce.merge(sht_agg, on=['Player_Clean', 'Born', 'Season'],
                   how='left', indicator='_m_sht')
print(base['_m_sht'].value_counts())
print('Filas:', len(cruce), '->', len(base))
print(base[['Sh_tot', 'SoT_tot']].isna().sum())
print('SoT > Sh (debe ser 0):', (base['SoT_tot'] > base['Sh_tot']).sum())

# Todo por 90 con una sola base de minutos (FBref)
n90 = base['Min'] / 90
base['Sh_p90'] = base['Sh_tot'] / n90
base['SoT_p90'] = base['SoT_tot'] / n90
base['npG_p90'] = base['NPG'] / n90
base['NPxG_p90'] = base['NPxG'] / n90
base['xA_p90'] = base['xA'] / n90
base['npG_menos_NPxG_p90'] = (base['NPG'] - base['NPxG']) / n90

crit = ['Sh_p90', 'SoT_p90', 'npG_p90', 'NPxG_p90', 'xA_p90', 'npG_menos_NPxG_p90']
print(base[crit].describe().round(2))

# Control con Salah
print(base.loc[base['Player_Clean'] == 'mohamed salah',
               ['Season', 'Min', 'Sh_tot', 'SoT_tot', 'npG_p90', 'NPxG_p90']].round(2).to_string())

_m_sht
both          2301
left_only        0
right_only       0
Name: count, dtype: int64
Filas: 2301 -> 2301
Sh_tot     0
SoT_tot    0
dtype: int64
SoT > Sh (debe ser 0): 0
        Sh_p90  SoT_p90  npG_p90  NPxG_p90   xA_p90  npG_menos_NPxG_p90
count  2301.00  2301.00  2099.00   2099.00  2099.00             2099.00
mean      2.38     0.94     0.28      0.30     0.15               -0.03
std       0.77     0.39     0.18      0.17     0.09                0.11
min       0.38     0.00     0.00      0.00     0.00               -0.51
25%       1.86     0.68     0.15      0.19     0.09               -0.09
50%       2.34     0.91     0.26      0.28     0.13               -0.02
75%       2.81     1.17     0.38      0.40     0.20                0.04
max       5.81     2.71     1.25      1.18     0.66                0.51
     Season   Min  Sh_tot  SoT_tot  npG_p90  NPxG_p90
1554  20-21  3078   126.0     54.0     0.47      0.46
1555  21-22  2762   139.0     60.0     0.59      0.65
1556  22-23  329

In [10]:
tm = '../datos/raw/transfermarkt/'
pl = pd.read_csv(tm + 'players.csv',
                 usecols=['player_id', 'name', 'date_of_birth', 'position', 'sub_position'])
pl['Born'] = pd.to_datetime(pl['date_of_birth'], errors='coerce').dt.year
pl['key'] = pl['name'].apply(limpiar_nombre)

# Nombre original de FBref (no el de Understat)
base['key'] = base['Player'].apply(limpiar_nombre)

ident = base[['key', 'Born']].drop_duplicates()
m = ident.merge(pl[['key', 'Born', 'player_id', 'name', 'position']],
                on=['key', 'Born'], how='left', indicator=True)

print(m['_merge'].value_counts())
print('Jugadores con más de un player_id:', m.duplicated(['key', 'Born']).sum())
print(m['position'].value_counts(dropna=False))
print(m[m['_merge'] == 'left_only'][['key', 'Born']].to_string())

_merge
both          780
left_only      54
right_only      0
Name: count, dtype: int64
Jugadores con más de un player_id: 2
position
Attack      695
Midfield     84
NaN          54
Defender      1
Name: count, dtype: int64
                          key    Born
13               adri embarba  1992.0
20         albert guðmundsson  1997.0
29           alexandre alemao  1998.0
33                      alfon  1999.0
40                      amath  1996.0
69             anthony lozano  1993.0
115              burak yılmaz  1985.0
130     cheikh tidiane sabaly  1999.0
150                     cucho  1999.0
168             dapo afolayan  1997.0
176        dehmaine assoumani  2005.0
184                     diego  2000.0
190          dion drena beljo  2002.0
206         eli junior kroupi  2006.0
209                elif elmas  1999.0
212            emanuel emegha  2003.0
214               emi buendia  1996.0
220  eric maxim choupo-moting  1989.0
229            ezequiel avila  1994.0
254        france

In [11]:
tm = '../datos/raw/transfermarkt/'

equivalencias_tm = {
    'emi buendia': 'emiliano buendia',
    'hwang hee-chan': 'hee-chan hwang',
    'mykhailo mudryk': 'mykhaylo mudryk',
    'oliver mcburnie': 'oli mcburnie',
    'son heung-min': 'heung-min son',
    'valentin castellanos': 'taty castellanos',
}

pl = pd.read_csv(tm + 'players.csv',
                 usecols=['player_id', 'name', 'date_of_birth', 'position', 'sub_position'])
pl['Born'] = pd.to_datetime(pl['date_of_birth'], errors='coerce').dt.year
pl['key'] = pl['name'].apply(limpiar_nombre)

base['key'] = base['Player'].apply(limpiar_nombre).replace(equivalencias_tm)

ident = base[['key', 'Born']].drop_duplicates()
m = ident.merge(pl[['key', 'Born', 'player_id', 'name', 'position']],
                on=['key', 'Born'], how='left', indicator=True)

print(m['_merge'].value_counts())
print('Jugadores con más de un player_id:', m.duplicated(['key', 'Born']).sum())
print(m['position'].value_counts(dropna=False))
print(m[m['_merge'] == 'left_only'][['key', 'Born']].to_string())

_merge
both          786
left_only      48
right_only      0
Name: count, dtype: int64
Jugadores con más de un player_id: 2
position
Attack      701
Midfield     84
NaN          48
Defender      1
Name: count, dtype: int64
                          key    Born
13               adri embarba  1992.0
20         albert guðmundsson  1997.0
29           alexandre alemao  1998.0
33                      alfon  1999.0
40                      amath  1996.0
69             anthony lozano  1993.0
115              burak yılmaz  1985.0
130     cheikh tidiane sabaly  1999.0
150                     cucho  1999.0
168             dapo afolayan  1997.0
176        dehmaine assoumani  2005.0
184                     diego  2000.0
190          dion drena beljo  2002.0
206         eli junior kroupi  2006.0
209                elif elmas  1999.0
212            emanuel emegha  2003.0
220  eric maxim choupo-moting  1989.0
229            ezequiel avila  1994.0
254        francesco esposito  2005.0
283     giorgos k

In [12]:
val = pd.read_csv(tm + 'player_valuations.csv',
                  usecols=['player_id', 'date', 'market_value_in_eur', 'current_club_name'])
val['date'] = pd.to_datetime(val['date']).astype('datetime64[ns]')
val['player_id'] = val['player_id'].astype('int64')
val = val.rename(columns={'current_club_name': 'club_tm'}).sort_values('date')

b = base.merge(m[['key', 'Born', 'player_id']], on=['key', 'Born'], how='left')
b['fecha_corte'] = pd.to_datetime('20' + b['Season'].str[-2:] + '-06-01').astype('datetime64[ns]')

con_id = b.dropna(subset=['player_id']).copy()
con_id['player_id'] = con_id['player_id'].astype('int64')
con_id = con_id.sort_values('fecha_corte')

costo = pd.merge_asof(con_id, val, left_on='fecha_corte', right_on='date',
                      by='player_id', direction='backward',
                      tolerance=pd.Timedelta(days=365))
costo['dias_desde_valoracion'] = (costo['fecha_corte'] - costo['date']).dt.days

print('Filas:', len(base), '->', len(costo))
print('Sin valoración:', costo['market_value_in_eur'].isna().sum())
print(costo['dias_desde_valoracion'].describe().round(0))
print(costo.loc[costo['key'] == 'mohamed salah',
                ['Season', 'Squad', 'club_tm', 'date', 'market_value_in_eur']].to_string())

Filas: 2301 -> 2200
Sin valoración: 12
count    2188.0
mean       98.0
std        72.0
min         0.0
25%         7.0
50%        77.0
75%       162.0
max       363.0
Name: dias_desde_valoracion, dtype: float64
     Season      Squad       club_tm       date  market_value_in_eur
102   20-21  Liverpool  Liverpool FC 2021-03-18          110000000.0
494   21-22  Liverpool  Liverpool FC 2021-12-23          100000000.0
908   22-23  Liverpool  Liverpool FC 2023-03-16           70000000.0
1298  23-24  Liverpool  Liverpool FC 2024-05-27           55000000.0
1810  24-25  Liverpool  Liverpool FC 2025-05-30           50000000.0
2121  25-26  Liverpool  Liverpool FC 2025-12-09           30000000.0


In [13]:
print(pl['sub_position'].value_counts().head(10))

costo = costo.merge(pl[['player_id', 'sub_position']].drop_duplicates('player_id'),
                    on='player_id', how='left')
costo['edad'] = 2000 + costo['Season'].str[-2:].astype(int) - costo['Born']

print(costo.loc[costo['Season'] == '25-26', 'sub_position'].value_counts())

sub_position
Centre-Back           8766
Centre-Forward        6749
Goalkeeper            5828
Central Midfield      5498
Defensive Midfield    4123
Right-Back            3700
Attacking Midfield    3520
Left-Back             3495
Left Winger           3297
Right Winger          3170
Name: count, dtype: int64
sub_position
Centre-Forward        186
Left Winger            70
Right Winger           58
Attacking Midfield     26
Second Striker         10
Central Midfield        6
Defensive Midfield      3
Right Midfield          2
Left Midfield           1
Name: count, dtype: int64


In [14]:
import numpy as np

# 1. Aislar el mercado actual (ej. temporada 25-26) para simular contrataciones reales
df_actual = costo[(costo['Season'] == '25-26') &
                  (costo['sub_position'].isin(['Centre-Forward', 'Second Striker']))].reset_index(drop=True)
print(len(df_actual), 'delanteros en 25-26')

# 2. Definir la matriz multicriterio
# Beneficios: Goles esperados sin penal, Tiros al arco, Asistencias esperadas
# Costo: Valor de mercado en Euros
criterios = ['NPxG_p90', 'SoT_p90', 'xA_p90', 'market_value_in_eur']

# Extraer la matriz de decisión (X)
X = df_actual[criterios].values

# 3. Parámetros del modelo
# Pesos: 40% Calidad de tiro, 15% Volumen, 15% Creación, 30% Costo económico
# (En un proyecto más avanzado, podrías usar AHP para calcular estos pesos exactos)
matriz_saaty = np.array([[1, 3, 3, 2], [1/3, 1, 1, 1/2], [1/3, 1, 1, 1/2], [1/2, 2, 2, 1]])
pesos_ahp = (matriz_saaty / matriz_saaty.sum(axis=0)).mean(axis=1)
pesos = pesos_ahp
impactos = np.array([1, 1, 1, -1]) # 1 = Maximizar (Beneficio), -1 = Minimizar (Costo)

# 4. Algoritmo TOPSIS Vectorial
# Normalización de la matriz
norm_X = X / np.sqrt((X**2).sum(axis=0))

# Matriz ponderada
V = norm_X * pesos

# Determinar las soluciones ideales (Mejor y Peor)
ideal_best = np.where(impactos == 1, V.max(axis=0), V.min(axis=0))
ideal_worst = np.where(impactos == 1, V.min(axis=0), V.max(axis=0))

# Calcular distancias euclidianas
dist_best = np.sqrt(((V - ideal_best)**2).sum(axis=1))
dist_worst = np.sqrt(((V - ideal_worst)**2).sum(axis=1))

# Calcular el puntaje final (Coeficiente de cercanía)
df_actual['Puntaje_TOPSIS'] = dist_worst / (dist_best + dist_worst)

# 5. Resultados: Los delanteros más eficientes del mercado
resultados_moneyball = df_actual.sort_values('Puntaje_TOPSIS', ascending=False)

print("🏆 Top 10 Delanteros Infravalorados (Alta Producción / Bajo Costo)")
columnas_vista = ['key', 'Squad', 'NPxG_p90', 'market_value_in_eur', 'Puntaje_TOPSIS']
display(resultados_moneyball[columnas_vista].head(10).round({'NPxG_p90': 2, 'Puntaje_TOPSIS': 4}))

196 delanteros en 25-26
🏆 Top 10 Delanteros Infravalorados (Alta Producción / Bajo Costo)


,key,Squad,NPxG_p90,market_value_in_eur,Puntaje_TOPSIS
0,akor adams,Sevilla,0.34,12000000.0,NaN
1,nikola krstovic,Atalanta,0.79,30000000.0,NaN
2,tim lemperle,Hoffenheim,0.40,17000000.0,NaN
3,taty castellanos,Lazio/West Ham,0.31,28000000.0,NaN
4,ivan romero,Levante,0.35,7500000.0,NaN
5,toni martinez,Alavés,NaN,4000000.0,NaN
6,nikola stulic,Lecce,0.25,3500000.0,NaN
7,emersonn,Toulouse,0.45,8000000.0,NaN
8,arnaud kalimuendo,Nottingham/Frankfurt,NaN,25000000.0,NaN
9,umar sadiq,Valencia/Real Sociedad,0.46,4000000.0,NaN


In [15]:
print(resultados_moneyball.head(10)[['key', 'Squad', 'edad', 'market_value_in_eur', 'Puntaje_TOPSIS']].to_string())

                 key                   Squad  edad  market_value_in_eur  Puntaje_TOPSIS
0         akor adams                 Sevilla  26.0           12000000.0             NaN
1    nikola krstovic                Atalanta  26.0           30000000.0             NaN
2       tim lemperle              Hoffenheim  24.0           17000000.0             NaN
3   taty castellanos          Lazio/West Ham  28.0           28000000.0             NaN
4        ivan romero                 Levante  25.0            7500000.0             NaN
5      toni martinez                  Alavés  29.0            4000000.0             NaN
6      nikola stulic                   Lecce  25.0            3500000.0             NaN
7           emersonn                Toulouse  22.0            8000000.0             NaN
8  arnaud kalimuendo    Nottingham/Frankfurt  24.0           25000000.0             NaN
9         umar sadiq  Valencia/Real Sociedad  29.0            4000000.0             NaN


In [16]:
import numpy as np

# ---------------------------------------------------------
# 1. AHP: CÁLCULO DE PESOS MEDIANTE MATRIZ DE SAATY
# Criterios: [NPxG_p90, SoT_p90, xA_p90, market_value_in_eur]
# ---------------------------------------------------------

# Matriz de comparaciones pareadas (Escala 1 al 9)
# Ej: NPxG es "moderadamente más importante" (3) que SoT y xA.
matriz_saaty = np.array([
    [1,   3,   3,   2],   # NPxG_p90
    [1/3, 1,   1,   1/2], # SoT_p90
    [1/3, 1,   1,   1/2], # xA_p90
    [1/2, 2,   2,   1]    # Valor de Mercado
])

# Normalización por columnas y extracción del vector propio (pesos)
matriz_norm = matriz_saaty / matriz_saaty.sum(axis=0)
pesos_ahp = matriz_norm.mean(axis=1)

print(f"📊 Pesos AHP calculados:\nNPxG: {pesos_ahp[0]:.1%} | SoT: {pesos_ahp[1]:.1%} | xA: {pesos_ahp[2]:.1%} | Costo: {pesos_ahp[3]:.1%}\n")

# ---------------------------------------------------------
# 2. MABAC: MULTI-ATTRIBUTIVE BORDER APPROXIMATION AREA
# ---------------------------------------------------------

# Identificar máximos y mínimos de la matriz de decisión (X, definida en tu código anterior)
X_min = X.min(axis=0)
X_max = X.max(axis=0)

# Normalización MABAC:
N = np.zeros_like(X)
# Para criterios de Beneficio (índices 0, 1, 2)
for j in [0, 1, 2]:
    N[:, j] = (X[:, j] - X_min[j]) / (X_max[j] - X_min[j])
# Para criterio de Costo (índice 3)
N[:, 3] = (X_max[3] - X[:, 3]) / (X_max[3] - X_min[3])

# Matriz ponderada añadiendo los pesos AHP
V_mabac = N * pesos_ahp + pesos_ahp

# Área de Aproximación Fronteriza (G) - Productoria de las columnas
G = np.prod(V_mabac, axis=0) ** (1 / len(V_mabac))

# Distancia de cada jugador al área fronteriza (Q)
Q = V_mabac - G

# Puntaje Final (Suma de las distancias Q)
df_actual['Puntaje_MABAC'] = Q.sum(axis=1)

# ---------------------------------------------------------
# 3. RESULTADOS Y VALIDACIÓN CRUZADA
# ---------------------------------------------------------

resultados_mabac = df_actual.sort_values('Puntaje_MABAC', ascending=False)

print("🏆 Top 10 Delanteros - Validación Cruzada (MABAC + AHP + TOPSIS)")
columnas_vista = ['key', 'Squad', 'NPxG_p90', 'market_value_in_eur', 'Puntaje_TOPSIS', 'Puntaje_MABAC']
display(resultados_mabac[columnas_vista].head(10).round({'NPxG_p90': 2, 'Puntaje_TOPSIS': 4, 'Puntaje_MABAC': 4}))

📊 Pesos AHP calculados:
NPxG: 45.5% | SoT: 14.1% | xA: 14.1% | Costo: 26.3%

🏆 Top 10 Delanteros - Validación Cruzada (MABAC + AHP + TOPSIS)


,key,Squad,NPxG_p90,market_value_in_eur,Puntaje_TOPSIS,Puntaje_MABAC
0,akor adams,Sevilla,0.34,12000000.0,NaN,NaN
1,nikola krstovic,Atalanta,0.79,30000000.0,NaN,NaN
2,tim lemperle,Hoffenheim,0.40,17000000.0,NaN,NaN
3,taty castellanos,Lazio/West Ham,0.31,28000000.0,NaN,NaN
4,ivan romero,Levante,0.35,7500000.0,NaN,NaN
5,toni martinez,Alavés,NaN,4000000.0,NaN,NaN
6,nikola stulic,Lecce,0.25,3500000.0,NaN,NaN
7,emersonn,Toulouse,0.45,8000000.0,NaN,NaN
8,arnaud kalimuendo,Nottingham/Frankfurt,NaN,25000000.0,NaN,NaN
9,umar sadiq,Valencia/Real Sociedad,0.46,4000000.0,NaN,NaN


In [17]:
lam = (matriz_saaty @ pesos_ahp / pesos_ahp).mean()
CR = ((lam - 4) / 3) / 0.90
print(f'Razón de consistencia: {CR:.3f}')

from scipy.stats import spearmanr
print(spearmanr(df_actual['Puntaje_TOPSIS'], df_actual['Puntaje_MABAC']))

Razón de consistencia: 0.004


SignificanceResult(statistic=nan, pvalue=nan)


In [18]:
import sys
!{sys.executable} -m pip install plotly streamlit

In [19]:
import plotly.express as px

# Gráfico de dispersión interactivo
fig = px.scatter(
    resultados_mabac, 
    x='market_value_in_eur', 
    y='NPxG_p90',
    hover_name='key',
    hover_data=['Squad', 'Puntaje_TOPSIS'],
    color='Puntaje_MABAC',
    size_max=15,
    title='Cuadrante Moneyball: Costo vs Producción Ofensiva (NPxG)',
    labels={'market_value_in_eur': 'Valor de Mercado (€)', 'NPxG_p90': 'Goles Esperados sin Penal (x90)'},
    template='plotly_dark'
)

# Invertimos el eje X para que los más baratos (mejores para Moneyball) queden a la derecha
fig.update_xaxes(autorange="reversed")
fig.show()

In [20]:
resultados_mabac.to_csv('../datos/procesado/resultados_finales.csv', index=False)

In [21]:
import numpy as np

# 1. Matriz de decisión (X) original
X_orig = df_actual[['NPxG_p90', 'SoT_p90', 'xA_p90', 'market_value_in_eur']].values

# TRUCO MATEMÁTICO: Sumamos 0.001 para evitar el colapso por "división por cero" en MEREC
X = X_orig + 0.001
X_min = X.min(axis=0)
X_max = X.max(axis=0)

# 2. Normalización específica de MEREC
N_merec = np.zeros_like(X)
for j in [0, 1, 2]: N_merec[:, j] = X_min[j] / X[:, j]  # Beneficios
N_merec[:, 3] = X[:, 3] / X_max[3]                      # Costo

# 3. Desempeño global (S_i) y efecto de remoción
S = np.log(1 + (1/4) * np.sum(np.abs(np.log(N_merec)), axis=1))

E = np.zeros(4)
for j in range(4):
    N_temp = np.delete(N_merec, j, axis=1)
    S_temp = np.log(1 + (1/3) * np.sum(np.abs(np.log(N_temp)), axis=1))
    E[j] = np.sum(np.abs(S_temp - S))

# 4. Pesos MEREC objetivos
pesos_merec = E / np.sum(E)
print(f"Pesos calculados por MEREC: NPxG {pesos_merec[0]:.1%} | SoT {pesos_merec[1]:.1%} | xA {pesos_merec[2]:.1%} | Costo {pesos_merec[3]:.1%}")

# 5. Puntaje MEREC Final (Normalizamos usando los datos originales, ya sin el 0.001)
X_min_orig = X_orig.min(axis=0)
X_max_orig = X_orig.max(axis=0)

N_std = np.zeros_like(X_orig)
for j in [0, 1, 2]: N_std[:, j] = (X_orig[:, j] - X_min_orig[j]) / (X_max_orig[j] - X_min_orig[j])
N_std[:, 3] = (X_max_orig[3] - X_orig[:, 3]) / (X_max_orig[3] - X_min_orig[3])

df_actual['Puntaje_MEREC'] = np.dot(N_std, pesos_merec)

# 6. Sobrescribir el CSV con la nueva columna calculada correctamente
df_actual.to_csv('../datos/procesado/resultados_finales.csv', index=False)

Pesos calculados por MEREC: NPxG nan% | SoT nan% | xA nan% | Costo nan%


In [22]:
def mcdm(df, criterios, impactos, pesos, norm='vector'):
    X = df[criterios].values.astype(float)
    imp = np.array(impactos)
    out = df.copy()
    lo, hi = X.min(axis=0), X.max(axis=0)

    # TOPSIS
    if norm == 'vector':
        R = X / np.sqrt((X**2).sum(axis=0))
    else:                                   # min-max: todos los criterios quedan en 0-1
        R = (X - lo) / (hi - lo)
    V = R * pesos
    best = np.where(imp == 1, V.max(axis=0), V.min(axis=0))
    worst = np.where(imp == 1, V.min(axis=0), V.max(axis=0))
    d_best = np.sqrt(((V - best)**2).sum(axis=1))
    d_worst = np.sqrt(((V - worst)**2).sum(axis=1))
    out['TOPSIS'] = d_worst / (d_best + d_worst)

    # MABAC (siempre min-max)
    N = np.where(imp == 1, (X - lo) / (hi - lo), (hi - X) / (hi - lo))
    Vm = N * pesos + pesos
    G = np.prod(Vm, axis=0) ** (1 / len(Vm))
    out['MABAC'] = (Vm - G).sum(axis=1)
    return out

In [24]:
def ahp(matriz):
    M = np.array(matriz)
    n = len(M)
    
    # 1. Normalizar y calcular los pesos (Vector Propio)
    pesos = (M / M.sum(axis=0)).mean(axis=1)
    
    # 2. Calcular el Índice de Consistencia (CI) y la Razón de Consistencia (CR)
    lam = (M @ pesos / pesos).mean()
    CI = (lam - n) / (n - 1)
    
    # Índices de Consistencia Aleatoria (RI) estándar de Saaty (para n de 1 a 10)
    RI_dict = {1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49}
    RI = RI_dict.get(n, 1.49)
    
    CR = CI / RI if RI > 0 else 0
    
    return pesos, CR

In [25]:
# Matriz de 4 criterios (la tuya): NPxG, SoT, xA, Valor
m4 = [[1, 3, 3, 2],
      [1/3, 1, 1, 1/2],
      [1/3, 1, 1, 1/2],
      [1/2, 2, 2, 1]]

# Matriz de 5 criterios: se agrega Edad (menos edad = mejor).
# Edad pesa como SoT y xA, la mitad que Valor y un tercio de NPxG.
m5 = [[1, 3, 3, 2, 3],
      [1/3, 1, 1, 1/2, 1],
      [1/3, 1, 1, 1/2, 1],
      [1/2, 2, 2, 1, 2],
      [1/3, 1, 1, 1/2, 1]]

w4, cr4 = ahp(m4)
w5, cr5 = ahp(m5)
print('Pesos 4 criterios:', w4.round(3), '| CR =', round(cr4, 3))
print('Pesos 5 criterios:', w5.round(3), '| CR =', round(cr5, 3))

crit4 = ['NPxG_p90', 'SoT_p90', 'xA_p90', 'market_value_in_eur']
imp4 = [1, 1, 1, -1]
crit5 = crit4 + ['edad']
imp5 = imp4 + [-1]

A = mcdm(df_actual, crit4, imp4, w4)                            # todos, 4 criterios
B = mcdm(df_actual[df_actual['edad'] <= 30], crit4, imp4, w4)   # solo 30 años o menos
C = mcdm(df_actual, crit5, imp5, w5)                            # edad como criterio

vista = ['key', 'Squad', 'edad', 'market_value_in_eur', 'TOPSIS', 'MABAC']
for nombre, r in [('A: todos, 4 criterios', A), ('B: 30 años o menos', B), ('C: edad como criterio', C)]:
    print(f'\n{nombre} ({len(r)} jugadores)')
    print(r.sort_values('TOPSIS', ascending=False).head(10)[vista].round(3).to_string())

print('\nSpearman TOPSIS vs MABAC  A:', round(spearmanr(A['TOPSIS'], A['MABAC'])[0], 2),
      '| B:', round(spearmanr(B['TOPSIS'], B['MABAC'])[0], 2),
      '| C:', round(spearmanr(C['TOPSIS'], C['MABAC'])[0], 2))
print('Spearman TOPSIS A vs C:', round(spearmanr(A['TOPSIS'], C['TOPSIS'])[0], 2))

Pesos 4 criterios: [0.455 0.141 0.141 0.263] | CR = 0.004
Pesos 5 criterios: [0.394 0.124 0.124 0.234 0.124] | CR = 0.002

A: todos, 4 criterios (196 jugadores)
                 key                   Squad  edad  market_value_in_eur  TOPSIS  MABAC
0         akor adams                 Sevilla  26.0           12000000.0     NaN    NaN
1    nikola krstovic                Atalanta  26.0           30000000.0     NaN    NaN
2       tim lemperle              Hoffenheim  24.0           17000000.0     NaN    NaN
3   taty castellanos          Lazio/West Ham  28.0           28000000.0     NaN    NaN
4        ivan romero                 Levante  25.0            7500000.0     NaN    NaN
5      toni martinez                  Alavés  29.0            4000000.0     NaN    NaN
6      nikola stulic                   Lecce  25.0            3500000.0     NaN    NaN
7           emersonn                Toulouse  22.0            8000000.0     NaN    NaN
8  arnaud kalimuendo    Nottingham/Frankfurt  24.0      

In [26]:
A2 = mcdm(df_actual, crit4, imp4, w4, norm='minmax')
C2 = mcdm(df_actual, crit5, imp5, w5, norm='minmax')

print('Spearman exacto A vs C, TOPSIS vectorial:', round(spearmanr(A['TOPSIS'], C['TOPSIS'])[0], 4))
print('Spearman TOPSIS vectorial vs min-max (4 criterios):', round(spearmanr(A['TOPSIS'], A2['TOPSIS'])[0], 3))
print('Spearman 4 vs 5 criterios, TOPSIS min-max:', round(spearmanr(A2['TOPSIS'], C2['TOPSIS'])[0], 3))
print('Spearman TOPSIS vs MABAC, 5 criterios, min-max:', round(spearmanr(C2['TOPSIS'], C2['MABAC'])[0], 3))

print('\nC: edad como criterio (TOPSIS min-max)')
print(C2.sort_values('TOPSIS', ascending=False).head(10)[vista].round(3).to_string())

Spearman exacto A vs C, TOPSIS vectorial: nan
Spearman TOPSIS vectorial vs min-max (4 criterios): nan
Spearman 4 vs 5 criterios, TOPSIS min-max: nan
Spearman TOPSIS vs MABAC, 5 criterios, min-max: nan

C: edad como criterio (TOPSIS min-max)
                 key                   Squad  edad  market_value_in_eur  TOPSIS  MABAC
0         akor adams                 Sevilla  26.0           12000000.0     NaN    NaN
1    nikola krstovic                Atalanta  26.0           30000000.0     NaN    NaN
2       tim lemperle              Hoffenheim  24.0           17000000.0     NaN    NaN
3   taty castellanos          Lazio/West Ham  28.0           28000000.0     NaN    NaN
4        ivan romero                 Levante  25.0            7500000.0     NaN    NaN
5      toni martinez                  Alavés  29.0            4000000.0     NaN    NaN
6      nikola stulic                   Lecce  25.0            3500000.0     NaN    NaN
7           emersonn                Toulouse  22.0            8

In [27]:
df_actual['log_valor'] = np.log10(df_actual['market_value_in_eur'])
crit4L = ['NPxG_p90', 'SoT_p90', 'xA_p90', 'log_valor']

L = mcdm(df_actual, crit4L, imp4, w4, norm='minmax')
print('Spearman TOPSIS vs MABAC (costo en log):', round(spearmanr(L['TOPSIS'], L['MABAC'])[0], 3))
print('Spearman contra min-max sin log:', round(spearmanr(L['TOPSIS'], A2['TOPSIS'])[0], 3))
print(L.sort_values('TOPSIS', ascending=False).head(10)[vista].round(3).to_string())

Spearman TOPSIS vs MABAC (costo en log): nan
Spearman contra min-max sin log: nan
                 key                   Squad  edad  market_value_in_eur  TOPSIS  MABAC
0         akor adams                 Sevilla  26.0           12000000.0     NaN    NaN
1    nikola krstovic                Atalanta  26.0           30000000.0     NaN    NaN
2       tim lemperle              Hoffenheim  24.0           17000000.0     NaN    NaN
3   taty castellanos          Lazio/West Ham  28.0           28000000.0     NaN    NaN
4        ivan romero                 Levante  25.0            7500000.0     NaN    NaN
5      toni martinez                  Alavés  29.0            4000000.0     NaN    NaN
6      nikola stulic                   Lecce  25.0            3500000.0     NaN    NaN
7           emersonn                Toulouse  22.0            8000000.0     NaN    NaN
8  arnaud kalimuendo    Nottingham/Frankfurt  24.0           25000000.0     NaN    NaN
9         umar sadiq  Valencia/Real Sociedad  29

In [30]:
import itertools
import pandas as pd

filas = []
for (nom, crit), norm, filtrar in itertools.product(
        [('eur', crit4), ('log', crit4L)], ['vector', 'minmax'], [False, True]):
    if nom == 'log' and norm == 'vector':
        continue                                  
    d = df_actual[df_actual['edad'] <= 30] if filtrar else df_actual
    r = mcdm(d, crit, imp4, w4, norm=norm)
    for metodo in ['TOPSIS', 'MABAC']:
        rk = r[metodo].rank(ascending=False)
        # Agregamos el equipo (Squad) para hacer el identificador aún más específico
        for k, s, sq, v in zip(r['key'], r['Season'], r['Squad'], rk):
            identificador = f"{k} ({s} | {sq})"
            filas.append((identificador, f"{nom}-{norm}-{'<=30' if filtrar else 'todos'}-{metodo}", v))

# Usamos pivot_table en lugar de pivot. 
# Si encuentra un duplicado, aggfunc='mean' promediará su posición en el ranking en lugar de arrojar error.
t = pd.DataFrame(filas, columns=['id', 'escenario', 'rank']).pivot_table(
    index='id', 
    columns='escenario', 
    values='rank',
    aggfunc='mean' 
)

resumen = pd.DataFrame({
    'veces_top5': (t <= 5).sum(axis=1),
    'rank_mediano': t.median(axis=1),
    'escenarios_con_dato': t.notna().sum(axis=1),
})
print(resumen.sort_values(['veces_top5', 'rank_mediano'], ascending=[False, True]).head(12))

Empty DataFrame
Columns: [veces_top5, rank_mediano, escenarios_con_dato]
Index: []


In [33]:
rng = np.random.default_rng(42)
N_SIM = 1000
ALPHA = 30        

escenarios = {
    'vector_todos':     (df_actual, crit4, 'vector'),
    'vector_<=30':      (df_actual[df_actual['edad'] <= 30], crit4, 'vector'),
    'minmax_log_todos': (df_actual, crit4L, 'minmax'),
    'minmax_log_<=30':  (df_actual[df_actual['edad'] <= 30], crit4L, 'minmax'),
}

res = {}
for nombre, (d, crit, norm) in escenarios.items():
    # Jaiporu index ha'eñóva ymave guaréicha
    d_unico = d.drop_duplicates(subset=['key', 'Season', 'Squad']).copy()
    ids_unicos = d_unico['key'] + ' (' + d_unico['Season'] + ' | ' + d_unico['Squad'] + ')'
    conteo = pd.Series(0.0, index=ids_unicos.values)
    
    for _ in range(N_SIM):
        w = rng.dirichlet(ALPHA * w4)                 
        r = mcdm(d_unico, crit, imp4, w, norm=norm)
        
        top_5 = r.nlargest(5, 'TOPSIS')
        top_5_ids = top_5['key'] + ' (' + top_5['Season'] + ' | ' + top_5['Squad'] + ')'
        conteo.loc[top_5_ids.values] += 1
        
    res[nombre] = conteo / N_SIM

tabla = pd.DataFrame(res)
tabla['promedio'] = tabla.mean(axis=1)
print((tabla.sort_values('promedio', ascending=False).head(12) * 100).round(0))

                                           vector_todos  vector_<=30  \
tim lemperle (25-26 | Hoffenheim)                 100.0        100.0   
akor adams (25-26 | Sevilla)                      100.0        100.0   
nikola krstovic (25-26 | Atalanta)                100.0        100.0   
ivan romero (25-26 | Levante)                     100.0        100.0   
taty castellanos (25-26 | Lazio/West Ham)         100.0        100.0   
adam armstrong (25-26 | Wolves)                     0.0          0.0   
mathys tel (25-26 | Tottenham)                      0.0          0.0   
matias fernandez-pardo (25-26 | Lille)              0.0          0.0   
matthis abline (25-26 | Nantes)                     0.0          0.0   
maximilian beier (25-26 | Dortmund)                 0.0          0.0   
michael gregoritsch (25-26 | Augsburg)              0.0          NaN   
mika biereth (25-26 | Monaco)                       0.0          0.0   

                                           minmax_log_todos  mi

In [34]:
import statsmodels.formula.api as smf

cf = costo[costo['sub_position'].isin(['Centre-Forward', 'Second Striker'])].copy()
cf['ln_valor'] = np.log(cf['market_value_in_eur'])
cf['edad2'] = cf['edad'] ** 2

modelo = smf.ols('ln_valor ~ NPxG_p90 + SoT_p90 + xA_p90 + edad + edad2 + Season', data=cf).fit()
print(modelo.summary().tables[1])
print('R2:', round(modelo.rsquared, 3), '| n =', int(modelo.nobs))

cf['valor_esperado_eur'] = np.exp(modelo.predict(cf))
cf['razon_valor'] = cf['market_value_in_eur'] / cf['valor_esperado_eur']    # menor a 1: cuesta menos de lo esperado

act = cf[cf['Season'] == '25-26'].copy()
cols = ['key', 'Squad', 'edad', 'NPxG_p90', 'market_value_in_eur', 'valor_esperado_eur', 'razon_valor']
print('\nMás baratos de lo esperado:')
print(act.sort_values('razon_valor').head(10)[cols].round(2).to_string())
print('\nMás caros de lo esperado:')
print(act.sort_values('razon_valor').tail(5)[cols].round(2).to_string())

                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          11.2931      0.954     11.835      0.000       9.421      13.165
Season[T.21-22]     0.0030      0.083      0.037      0.971      -0.160       0.166
Season[T.22-23]    -0.1095      0.083     -1.318      0.188      -0.273       0.054
Season[T.23-24]    -0.1494      0.086     -1.743      0.082      -0.318       0.019
Season[T.24-25]    -0.1071      0.088     -1.217      0.224      -0.280       0.066
Season[T.25-26]    -0.0377      0.085     -0.442      0.659      -0.205       0.130
NPxG_p90            0.8034      0.205      3.923      0.000       0.402       1.205
SoT_p90             1.0295      0.092     11.235      0.000       0.850       1.209
xA_p90              2.6307      0.333      7.907      0.000       1.978       3.283
edad                0.3526      0.067      5.277      0.000       0.221     

In [36]:
# 1. Definimos las columnas exactas que usa el modelo y el ID
columnas_modelo = ['ln_valor', 'NPxG_p90', 'SoT_p90', 'xA_p90', 'edad', 'edad2', 'Season', 'player_id', 'Squad', 'market_value_in_eur']

# 2. Eliminamos las filas que tengan nulos en estas variables ANTES de la regresión
cf_limpio = cf.dropna(subset=columnas_modelo).copy()

# 3. Ahora ejecutamos los modelos usando cf_limpio (así las longitudes coinciden perfecto)
m1 = smf.ols('ln_valor ~ NPxG_p90 + SoT_p90 + xA_p90 + edad + edad2 + Season', data=cf_limpio).fit(
        cov_type='cluster', cov_kwds={'groups': cf_limpio['player_id']})
print(m1.summary().tables[1])

m2 = smf.ols('ln_valor ~ NPxG_p90 + SoT_p90 + xA_p90 + edad + edad2 + Season + Squad', data=cf_limpio).fit()
print('\nR2 con club:', round(m2.rsquared, 3))

# 4. Calculamos la razón de valor usando la misma base limpia
cf_limpio['razon_club'] = cf_limpio['market_value_in_eur'] / np.exp(m2.predict(cf_limpio))
act = cf_limpio[cf_limpio['Season'] == '25-26'].copy()

# Traemos 'razon_valor' de la celda anterior cruzando por el índice para evitar errores
act['razon_valor'] = cf.loc[act.index, 'razon_valor']

print("\nLos 10 delanteros con mayor 'descuento' según su rendimiento, edad y club:")
print(act.sort_values('razon_club').head(10)[
    ['key', 'Squad', 'edad', 'NPxG_p90', 'market_value_in_eur', 'razon_valor', 'razon_club']].round(2).to_string())

                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          11.2931      1.209      9.342      0.000       8.924      13.662
Season[T.21-22]     0.0030      0.065      0.047      0.963      -0.125       0.131
Season[T.22-23]    -0.1095      0.071     -1.540      0.124      -0.249       0.030
Season[T.23-24]    -0.1494      0.078     -1.913      0.056      -0.302       0.004
Season[T.24-25]    -0.1071      0.083     -1.288      0.198      -0.270       0.056
Season[T.25-26]    -0.0377      0.086     -0.436      0.663      -0.207       0.132
NPxG_p90            0.8034      0.269      2.987      0.003       0.276       1.331
SoT_p90             1.0295      0.120      8.610      0.000       0.795       1.264
xA_p90              2.6307      0.380      6.923      0.000       1.886       3.375
edad                0.3526      0.085      4.134      0.000       0.185     

In [38]:
print('Modelo con club: R2 ajustado =', round(m2.rsquared_adj, 3), '| parámetros =', int(m2.df_model) + 1)

conteo = cf_limpio['Squad'].value_counts()
cf_limpio['club_g'] = cf_limpio['Squad'].where(cf_limpio['Squad'].map(conteo) >= 8, 'otros')
m3 = smf.ols('ln_valor ~ NPxG_p90 + SoT_p90 + xA_p90 + edad + edad2 + Season + club_g', data=cf_limpio).fit()
print('Clubes agrupados: R2 =', round(m3.rsquared, 3), '| R2 ajustado =', round(m3.rsquared_adj, 3),
      '| parámetros =', int(m3.df_model) + 1)

cf_limpio['razon_club_g'] = cf_limpio['market_value_in_eur'] / np.exp(m3.predict(cf_limpio))

# Usamos cf_limpio para filtrar la temporada actual
act = cf_limpio[cf_limpio['Season'] == '25-26']
sel = act[act['key'].isin(['lukas nmecha', 'callum wilson', 'beto', 'ollie watkins'])]
print(sel[['key', 'Squad', 'market_value_in_eur', 'razon_valor', 'razon_club', 'razon_club_g']].round(2).to_string())

Modelo con club: R2 ajustado = 0.791 | parámetros = 241
Clubes agrupados: R2 = 0.631 | R2 ajustado = 0.606 | parámetros = 73
                key         Squad  market_value_in_eur  razon_valor  razon_club  razon_club_g
1865  callum wilson      West Ham            5000000.0         1.00        0.66          1.22
1929  ollie watkins   Aston Villa           30000000.0         2.96        1.15          3.94
2006           beto       Everton           20000000.0         1.75        0.89          0.91
2027   lukas nmecha  Leeds United           10000000.0         0.55        0.50          0.73


In [39]:
sd = np.sqrt(m3.scale)
print('Desviación residual (log):', round(sd, 2))

sel = act[act['key'].isin(['lukas nmecha', 'callum wilson', 'beto', 'ollie watkins'])].copy()
sel['z'] = np.log(sel['razon_club_g']) / sd
print(sel[['key', 'Min', 'dias_desde_valoracion', 'razon_club_g', 'z']].round(2).to_string())

Desviación residual (log): 0.71
                key   Min  dias_desde_valoracion  razon_club_g     z
1865  callum wilson  1251                  174.0          1.22  0.28
1929  ollie watkins  2839                  174.0          3.94  1.92
2006           beto  1568                  174.0          0.91 -0.13
2027   lukas nmecha  1083                   84.0          0.73 -0.44
